# LG Aimers 데이터 품질 점검

## tl;dr

- 메인 학습 데이터 1,475,092행은 `row_id` 중복과 핵심 도메인 규칙 위반이 0건이다.
- 타깃률은 2019년 56.47%에서 2024년 48.61%로 하락해 시간 기반 검증이 필수다.
- Trackman은 97개 카운트 예외가 있으며, 메인 투수 ID와 직접 겹치는 ID가 0개라 선수 단위 조인 전에 매핑 근거가 필요하다.


## Context & Methods

투구 단위 확률 모델과 오프라인 제출 파이프라인에 사용하기 전에 스키마, 완전성, 고유성, 도메인 유효성, 시즌별 변화와 Trackman 연결 가능성을 청크 단위로 검사한다.

### Key Assumptions

- `row_id`와 `trackman_id`는 각 파일의 행 식별자다.
- 메인 데이터의 카운트·주자·점수·공식 `asof_*` 범위는 데이터 설명서 정의를 따른다.
- Trackman 날짜는 `M/D/YYYY`와 `YYYY-MM-DD` 혼합 형식을 허용한다.
- 배포 `test.csv` 5행은 형식 확인용이므로 2025 분포를 대표한다고 가정하지 않는다.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.lg_aimers.data_quality import run_data_quality_audit

DATA_DIR = ROOT / 'data'
report = run_data_quality_audit(DATA_DIR)
print(f"source={DATA_DIR}")


source=/Users/suyeong/Projects/personal/LG_Aimers/data


## Data


In [2]:
pd.DataFrame({
    'dataset': ['train', 'test_sample', 'trackman'],
    'rows': [report['train']['rows'], report['test_sample']['rows'], report['trackman']['rows']],
    'columns': [report['train']['columns'], report['test_sample']['columns'], report['trackman']['columns']],
})


,dataset,rows,columns
0,train,1475092,49
1,test_sample,5,48
2,trackman,1793078,30


## Results

### 1. 시즌별 크기와 타깃률


In [3]:
season_summary = pd.DataFrame.from_dict(report['train']['seasonal'], orient='index')
season_summary.index.name = 'season'
season_summary


,rows,target_rate,missing_asof_cells
season,,,
2019,237413,0.564670,77308
2020,244087,0.532712,24260
2021,247088,0.532762,23382
2022,247472,0.528920,18526
2023,245525,0.499957,16978
2024,253507,0.486105,22652


### 2. 메인 데이터 유효성·결측


In [4]:
main_summary = {
    'schema_matches': report['schema']['train_test_feature_schema_matches'],
    'duplicate_row_ids': report['train']['duplicate_row_ids'],
    'target_rate': report['train']['target_rate'],
    'domain_violations': sum(report['train']['invalid_counts'].values()),
}
display(pd.Series(main_summary, name='value').to_frame())
display(pd.Series(report['train']['null_rates'], name='null_rate').sort_values(ascending=False).head(15).to_frame())


,value
schema_matches,True
duplicate_row_ids,0
target_rate,0.523766
domain_violations,0


,null_rate
asof_pitcher_prev5_game_success_rate,0.019785
asof_pitcher_prev3_game_middle_rate,0.019785
asof_pitcher_prev1_game_success_rate,0.019785
asof_pitcher_prev3_game_success_rate,0.019785
asof_pitcher_prev1_game_middle_rate,0.019785
asof_pitcher_prev5_game_middle_rate,0.019785
asof_batter_middle_rate,0.000563
asof_batter_success_rate,0.000563
asof_pitcher_success_rate,0.000537
asof_pitcher_fastball_rate,0.000537


### 3. Trackman 품질과 연결성


In [5]:
trackman_summary = {
    'rows': report['trackman']['rows'],
    'duplicate_trackman_ids': report['trackman']['duplicate_trackman_ids'],
    'date_range': f"{report['trackman']['date_min']} ~ {report['trackman']['date_max']}",
    'count_rule_exceptions': sum(report['trackman']['invalid_counts'].values()),
    'direct_pitcher_id_overlap': report['linkage']['direct_pitcher_id_overlap'],
}
display(pd.Series(trackman_summary, name='value').to_frame())
pd.Series(report['trackman']['invalid_counts'], name='count').to_frame()


,value
rows,1793078
duplicate_trackman_ids,0
date_range,2019-03-23 ~ 2024-10-17
count_rule_exceptions,97
direct_pitcher_id_overlap,0


,count
invalid_game_date,0
season_date_mismatch,0
month_date_mismatch,0
dayofweek_date_mismatch,0
balls_outside_0_3,1
strikes_outside_0_2,1
outs_outside_0_2,95
pitch_no_not_positive,0


## Takeaways

1. 메인 데이터는 기준선 학습에 사용할 수 있는 품질이다. 공식 `asof_*` 결측은 cold-start로 보고 fold 학습 구간의 중앙값 또는 명시적 fallback으로 처리한다.
2. 타깃률의 강한 연도 하락 때문에 랜덤 분할 결과는 의사결정에 사용하지 않는다. 2022·2023·2024 forward split을 고정하고 평균 Brier와 편차를 함께 기록한다.
3. Trackman의 97개 비정상 카운트 행은 파생 집계 전에 제외한다.
4. 메인 `pitcher_id`와 `pitcher_trackman_id`는 직접 조인하지 않는다. 공식 매핑 또는 검증 가능한 연결 키가 확보되기 전에는 Trackman 선수별 피처를 보류한다.
